In [9]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [11]:
# Discover all folders in ADLS golddata container
from pyspark.sql import SparkSession

CONTAINER_PATH = "abfss://golddata@clientdatastorage.dfs.core.windows.net/"
DB_GOLD = "bupa_gold"

spark = SparkSession.builder.getOrCreate()

# Discover folders by reading binary files recursively
df_files = spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(CONTAINER_PATH)

# Extract unique top-level folder names
folder_set = set()
for row in df_files.select("path").distinct().collect():
    path = row.path.replace(CONTAINER_PATH, "").split("/")[0]
    if path and not path.startswith("_"):
        folder_set.add(path)

folders = sorted(list(folder_set))
print(f"✅ Found {len(folders)} folders in golddata container\n")
for i, folder in enumerate(folders, 1):
    print(f"{i}. {folder}")


✅ Found 35 folders in golddata container

1. dim_channel
2. dim_claim_type
3. dim_member_segment
4. dim_product_line
5. dim_providers
6. dim_region
7. dm_claims_experience
8. dm_member_value
9. dm_policy_retention
10. dq_monitoring
11. fact_claims
12. fact_members
13. fact_policies
14. ft_claims_risk
15. ft_claims_risk_split
16. ft_policy_churn
17. ft_policy_churn_split
18. ml_monitoring
19. ml_monitoring_view
20. models
21. scored_claims
22. scored_policy_churn
23. star_claims
24. star_members
25. star_policies
26. vw_claims_experience_kpi
27. vw_member_profile
28. vw_member_value_kpi
29. vw_ml_claims_risk_features
30. vw_ml_policy_churn_features
31. vw_policy_portfolio
32. vw_policy_retention_kpi
33. vw_scored_claims_fraud
34. vw_scored_claims_high_cost
35. vw_scored_policy_churn


In [ ]:
# Detect data format, convert to Delta, and load to database
from pathlib import Path
from pyspark.sql import SparkSession
import os

CONTAINER_PATH = "abfss://golddata@clientdatastorage.dfs.core.windows.net/"
DB_GOLD = "bupa_gold"
GOLD_DB_PATH = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/DataBase/bupa_gold"
DELTA_STAGING = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_03_Gold/delta_staging"

spark = SparkSession.builder.getOrCreate()

# Create database with persistent location
Path(GOLD_DB_PATH).mkdir(parents=True, exist_ok=True)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD} LOCATION '{GOLD_DB_PATH}'")
print(f"✅ Database created: {DB_GOLD} at {GOLD_DB_PATH}\n")

# Discover all folders
df_files = spark.read.format("binaryFile").option("recursiveFileLookup", "true").load(CONTAINER_PATH)

folder_set = set()
for row in df_files.select("path").distinct().collect():
    path = row.path.replace(CONTAINER_PATH, "").split("/")[0]
    if path and not path.startswith("_"):
        folder_set.add(path)

folders = sorted(list(folder_set))
print(f"Loading {len(folders)} tables (auto-detecting format)...\n")

loaded_count = 0
for idx, folder in enumerate(folders, 1):
    folder_path = f"{CONTAINER_PATH}{folder}/"
    table_name = folder.replace('-', '_').replace(' ', '_').lower()
    table_full_name = f"{DB_GOLD}.{table_name}"
    staging_path = f"{DELTA_STAGING}/{table_name}"
    
    try:
        # Auto-detect and read format (Delta or Parquet)
        df = None
        try:
            # Try Delta first
            df = spark.read.format("delta").load(folder_path)
            format_type = "Delta"
        except:
            # Fall back to Parquet
            try:
                df = spark.read.format("parquet").load(folder_path)
                format_type = "Parquet"
            except:
                # Fall back to any format
                df = spark.read.load(folder_path)
                format_type = "Auto"
        
        row_count = df.count()
        
        # Convert to Delta format in staging area
        Path(DELTA_STAGING).mkdir(parents=True, exist_ok=True)
        df.write.format("delta").mode("overwrite").save(staging_path)
        
        # Drop old table if exists
        spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")
        
        # Create table from Delta staging
        spark.sql(f"CREATE TABLE {table_full_name} USING DELTA LOCATION '{staging_path}'")
        
        print(f"[{idx}/{len(folders)}] ✅ {table_name:30s} - {row_count:,} rows ({format_type})")
        loaded_count += 1
        
    except Exception as e:
        print(f"[{idx}/{len(folders)}] ⚠️ {table_name:30s} - Error: {str(e)[:60]}")

print(f"\n{'='*70}")
print(f"Successfully loaded {loaded_count}/{len(folders)} tables into {DB_GOLD}")
print(f"{'='*70}\n")

# Show all tables
tables = spark.sql(f"SHOW TABLES IN {DB_GOLD}").collect()
print(f"📊 Tables in {DB_GOLD}: {len(tables)}\n")
for table in tables:
    print(f"  - {table.tableName}")

✅ Database created: bupa_gold at /Users/manojrammopati/Public/Projects/bupa_insurance_project/DataBase/bupa_gold



ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/spark_local/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
 

Loading 35 tables (auto-detecting format)...



[1/35] ✅ dim_channel                    - 40 rows (Parquet)
[2/35] ✅ dim_claim_type                 - 6 rows (Delta)
[2/35] ✅ dim_claim_type                 - 6 rows (Delta)


In [7]:
# Make tables accessible from other notebooks
from pyspark.sql import SparkSession

DB_GOLD = "bupa_gold"
spark = SparkSession.builder.getOrCreate()

# Set default database
spark.sql(f"USE {DB_GOLD}")

# List all available tables
tables_df = spark.sql(f"SHOW TABLES IN {DB_GOLD}")
tables = [row.tableName for row in tables_df.collect()]

print(f"✅ Database '{DB_GOLD}' is now active\n")
print(f"📊 Available tables ({len(tables)}):\n")
for i, table in enumerate(tables, 1):
    # Get row count for each table
    try:
        row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {DB_GOLD}.{table}").collect()[0][0]
        print(f"   {i}. {table:35s} ({row_count:,} rows)")
    except:
        print(f"   {i}. {table:35s} (rows unknown)")

print(f"\n{'='*70}")
print("💡 To access these tables in OTHER NOTEBOOKS:")
print(f"{'='*70}")
print(f"""
# In any other notebook, add this at the start:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sql("USE {DB_GOLD}")

# Then query any table directly:
spark.sql("SELECT * FROM <table_name> LIMIT 5").show()

# Or in SQL cells:
%sql
SELECT * FROM {DB_GOLD}.<table_name> LIMIT 10
""")
print(f"{'='*70}\n")

✅ Database 'bupa_gold' is now active

📊 Available tables (33):



   1. dim_channel                         (4 rows)
   2. dim_claim_type                      (6 rows)
   2. dim_claim_type                      (6 rows)
   3. dim_member_segment                  (1,440 rows)
   3. dim_member_segment                  (1,440 rows)
   4. dim_product_line                    (5 rows)
   4. dim_product_line                    (5 rows)
   5. dim_providers                       (5,393 rows)
   5. dim_providers                       (5,393 rows)


   6. dim_region                          (53 rows)
   7. dm_claims_experience                (558,211 rows)
   7. dm_claims_experience                (558,211 rows)
   8. dm_member_value                     (381,109 rows)
   8. dm_member_value                     (381,109 rows)


   9. dm_policy_retention                 (183 rows)


   10. dq_monitoring                       (24 rows)
   11. fact_claims                         (558,211 rows)
   11. fact_claims                         (558,211 rows)


   12. fact_members                        (381,109 rows)
   13. fact_policies                       (381,109 rows)
   13. fact_policies                       (381,109 rows)
   14. ft_claims_risk                      (558,211 rows)
   14. ft_claims_risk                      (558,211 rows)
   15. ft_claims_risk_split                (558,211 rows)
   15. ft_claims_risk_split                (558,211 rows)
   16. ft_policy_churn                     (381,109 rows)
   16. ft_policy_churn                     (381,109 rows)
   17. ft_policy_churn_split               (381,109 rows)
   17. ft_policy_churn_split               (381,109 rows)
   18. ml_monitoring                       (29 rows)
   18. ml_monitoring                       (29 rows)
   19. ml_monitoring_view                  (558,211 rows)
   19. ml_monitoring_view                  (558,211 rows)
   20. scored_policy_churn                 (314,128 rows)
   20. scored_policy_churn                 (314,128 rows)
   21. star_claims      

   28. vw_ml_policy_churn_features         (381,109 rows)
   29. vw_policy_portfolio                 (381,109 rows)
   29. vw_policy_portfolio                 (381,109 rows)
   30. vw_policy_retention_kpi             (183 rows)
   30. vw_policy_retention_kpi             (183 rows)
   31. vw_scored_claims_fraud              (558,211 rows)
   31. vw_scored_claims_fraud              (558,211 rows)
   32. vw_scored_claims_high_cost          (558,211 rows)
   32. vw_scored_claims_high_cost          (558,211 rows)
   33. vw_scored_policy_churn              (314,128 rows)

💡 To access these tables in OTHER NOTEBOOKS:

# In any other notebook, add this at the start:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sql("USE bupa_gold")

# Then query any table directly:
spark.sql("SELECT * FROM <table_name> LIMIT 5").show()

# Or in SQL cells:
%sql
SELECT * FROM bupa_gold.<table_name> LIMIT 10


   33. vw_scored_policy_churn              (314,128 rows)

💡 To

In [8]:
# Universal helper: Setup gold database access (use this in ANY notebook)
from pyspark.sql import SparkSession

def setup_gold_database():
    """Initialize connection to bupa_gold database"""
    spark = SparkSession.builder.getOrCreate()
    DB_GOLD = "bupa_gold"
    
    # Use the database
    spark.sql(f"USE {DB_GOLD}")
    
    # Verify it exists
    try:
        tables = spark.sql(f"SHOW TABLES IN {DB_GOLD}").count()
        print(f"✅ Connected to '{DB_GOLD}' database ({tables} tables available)")
        return spark, DB_GOLD
    except Exception as e:
        print(f"⚠️ Error connecting to {DB_GOLD}: {e}")
        return None, None

# Initialize connection
spark, DB_GOLD = setup_gold_database()

# Now you can query tables directly
print(f"\n💡 Example queries:\n")
print(f"spark.sql('SELECT * FROM dim_channel LIMIT 5').show()")
print(f"spark.sql('SELECT * FROM fact_claims LIMIT 5').show()")
print(f"spark.sql('SELECT COUNT(*) FROM fact_policies').show()")

✅ Connected to 'bupa_gold' database (33 tables available)

💡 Example queries:

spark.sql('SELECT * FROM dim_channel LIMIT 5').show()
spark.sql('SELECT * FROM fact_claims LIMIT 5').show()
spark.sql('SELECT COUNT(*) FROM fact_policies').show()
